# 13 — LLMs abertas (Qwen2.5-7B + Llama-3.1-8B) — binario + multiclasse

Roda 2 LLMs x 2 tarefas = 4 result_cards no test set.

**Setup necessario antes**:
1. **Llama-3.1 e gated** — aceite a licenca em https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
2. **Adicione HF_TOKEN no Colab Secrets**: clique no icone de chave na barra lateral esquerda > Add new secret > Name=`HF_TOKEN` Value=`hf_...` > toggle 'Notebook access' ON.
3. **L4 com 24 GB VRAM** (ou A100). Em T4, troque LOAD_DTYPE pra 'int8'.

**Tempos esperados em L4** (FP16, batch_size=8):
- `MAX_SAMPLES=200` (smoke test default): ~30s por modelo por tarefa = ~2 min total
- `MAX_SAMPLES=None` (test set completo, 16k): ~50min por modelo por tarefa = ~3h30 total

## 0. Verificacao de GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU nao detectada. Va em Runtime > Change runtime type > GPU (L4 ou A100).",
    )

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"GPU: {GPU_NAME} ({VRAM_GB} GB VRAM)")
print(f"CUDA: {torch.version.cuda}")

HARDWARE = f"Colab-{GPU_NAME.split()[-1]}"

if VRAM_GB < 20:
    print(f"\nAVISO: {GPU_NAME} tem pouca VRAM ({VRAM_GB} GB). "
          f"Considere LOAD_DTYPE='int8' ou USE_4BIT.")


## 1. Bootstrap (Colab + Local)

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Python   :", sys.version.split()[0])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )
    _run(
        [sys.executable, "-m", "pip", "install",
         "transformers>=4.45", "accelerate>=0.34", "-q"],
        "pip install transformers/accelerate",
    )

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "test.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), f"Falta colab_splits.zip em {DRIVE_BASE}."
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")

    RUNS_BASE = DRIVE_BASE / "runs"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_BASE = REPO_DIR / "artifacts" / "runs"

RUNS_BASE.mkdir(parents=True, exist_ok=True)
print("SPLITS_DIR:", SPLITS_DIR)
print("RUNS_BASE :", RUNS_BASE)
print("HARDWARE  :", HARDWARE)


def _sanity_check_imports() -> None:
    failures = []
    for mod_name in ("numpy", "scipy", "pandas", "torch", "transformers", "economy_classifier"):
        try:
            __import__(mod_name)
        except Exception as e:  # noqa: BLE001
            failures.append((mod_name, type(e).__name__, str(e)))
    if failures:
        for name, kind, msg in failures:
            print(f"  - {name}: {kind}: {msg[:200]}")
        raise RuntimeError(
            "Modulos quebrados. Solucao: Runtime > Restart runtime, e re-execute."
        )
    import torch, transformers  # noqa: E401
    print(f"\ntorch={torch.__version__}  transformers={transformers.__version__}")
    print("OK: economy_classifier importavel")


_sanity_check_imports()


## 2. Autenticacao HuggingFace

Le `HF_TOKEN` dos Colab Secrets (icone de chave na sidebar). Localmente usa `os.environ['HF_TOKEN']`.

In [ ]:
# Llama-3.1-8B-Instruct e gated. Token lido do Colab Secrets (chave HF_TOKEN).
# Localmente, define a env var antes: export HF_TOKEN=hf_...

HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("HF_TOKEN carregado de Colab Secrets.")
    except Exception as e:
        print(f"Nao consegui ler HF_TOKEN de userdata: {e}")
        HF_TOKEN = os.environ.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    print("AVISO: HF_TOKEN nao encontrado. Llama-3.1 (gated) vai falhar com GatedRepoError.")
    print("       Qwen2.5 (Apache 2.0) roda sem token.")
else:
    # Login programatico — evita prompt interativo
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Login HuggingFace OK.")


## 3. Imports e configuracao

In [ ]:
import gc
import json
import time
from pathlib import Path

import pandas as pd
import torch

from economy_classifier.datasets import MULTICLASS_TOP7, OTHER_LABEL, attach_multiclass_label
from economy_classifier.evaluation import (
    compute_binary_metrics,
    compute_confusion_matrix,
    compute_multiclass_metrics,
    compute_cost_metrics,
    compute_roc_auc,
)
from economy_classifier.llm_review import (
    LLM_REGISTRY,
    VALID_MULTI_LABELS,
    build_review_prompt,
    build_review_prompt_multiclass,
    classify_batch_hf,
    hf_results_to_multiclass_predictions,
    hf_results_to_predictions,
    load_hf_model,
    parse_llm_response,
    parse_llm_response_multiclass,
)
from economy_classifier.project import (
    build_result_card,
    persist_result_card,
)

SEED = 42
MULTI_LABELS = list(MULTICLASS_TOP7) + [OTHER_LABEL]

# ---- Quais (modelo x tarefa) rodar ----
LLMS = [
    ("qwen2.5-7b-instruct",   LLM_REGISTRY["qwen2.5-7b-instruct"]),
    ("llama-3.1-8b-instruct", LLM_REGISTRY["llama-3.1-8b-instruct"]),
]
TASKS = ["binary", "multiclass"]

# ---- Configuracao da inferencia ----
LOAD_DTYPE       = "float16"
BATCH_SIZE       = 8
MAX_NEW_TOKENS   = 100
MAX_INPUT_LENGTH = 2048
TEMPERATURE      = 0.0

# ---- Sampling ----
# 200 = smoke test rapido (~30s/modelo). Para producao, troque por None ou 16629.
MAX_SAMPLES = 200

print(f"LLMs        : {[k for k, _ in LLMS]}")
print(f"Tarefas     : {TASKS}")
print(f"BATCH_SIZE  : {BATCH_SIZE}")
print(f"MAX_SAMPLES : {MAX_SAMPLES}  ({'smoke test' if MAX_SAMPLES else 'test set completo'})")


## 4. Carga do test set

Subsample estratificado opcional via `MAX_SAMPLES`. `label_multi` (top-7+other) anexado se faltar.

In [ ]:
test = pd.read_parquet(SPLITS_DIR / "test.parquet")
if "label_multi" not in test.columns:
    test = attach_multiclass_label(test)

print(f"test: {len(test):,} registros (mercado={test['label'].mean()*100:.2f}%)")

if MAX_SAMPLES is not None and MAX_SAMPLES < len(test):
    # Estratificado por label binario (mantem ~12.5% mercado)
    test_eval = (
        test.groupby("label", group_keys=False)
            .apply(lambda g: g.sample(n=int(MAX_SAMPLES * len(g) / len(test)),
                                       random_state=SEED))
            .reset_index(drop=False)  # keep original index in column
    )
    test_eval = test_eval.set_index("index")
    print(f"Subsample: {len(test_eval):,} registros "
          f"(mercado={test_eval['label'].mean()*100:.2f}%)")
else:
    test_eval = test.copy()
    print(f"Avaliando o test set completo ({len(test_eval):,} registros).")

RECORD_IDS  = test_eval.index.tolist()
TEXTS       = test_eval["text"].fillna("").tolist()
Y_TRUE_BIN  = test_eval["label"].values
Y_TRUE_MULTI = test_eval["label_multi"].values


## 5. Funcao de avaliacao por (modelo x tarefa)

`evaluate_llm(model_key, model_name, task)`:
- `task='binary'`: prompt + parser binario, `y_score` 0/1, AUC-ROC
- `task='multiclass'`: prompt com 8 categorias, parser multiclasse, macro_F1 + matriz de confusao

Em ambos: zero-shot, `temperature=0` (greedy), checkpoint a cada 200 records.

In [ ]:
def evaluate_llm(model_key: str, model_name: str, task: str) -> dict:
    """Carrega modelo, classifica para *task*, salva predictions + result_card.

    task in {"binary", "multiclass"}.
    Retorna sumario com metricas e custo.
    """
    if task not in {"binary", "multiclass"}:
        raise ValueError(f"task deve ser binary ou multiclass, got {task!r}")

    model_id = f"llm_{model_key.replace('.', '_').replace('-', '_')}"
    print(f"\n{'='*72}\nLLM: {model_id}  task={task}  ({model_name})\n{'='*72}")

    run_dir = RUNS_BASE / f"{model_id}_{task}_test_set"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = run_dir / "predictions_checkpoint.csv"

    # ---- Carga (se ainda nao carregado para este modelo) ----
    print(f"[{model_id}] Carregando modelo (dtype={LOAD_DTYPE})...")
    t0 = time.perf_counter()
    tokenizer, model = load_hf_model(model_name, dtype=LOAD_DTYPE)
    load_time = time.perf_counter() - t0

    n_params = sum(p.numel() for p in model.parameters())
    model_size_mb = round(n_params * (2 if LOAD_DTYPE == "float16" else 1) / 1e6, 1)
    print(f"[{model_id}] Carregado em {load_time:.1f}s. "
          f"Params={n_params/1e9:.2f}B  Tamanho={model_size_mb} MB")

    # ---- Selecao de prompt e parser ----
    if task == "binary":
        prompt_builder = build_review_prompt
        parser = parse_llm_response
        Y_TRUE = Y_TRUE_BIN
    else:
        prompt_builder = build_review_prompt_multiclass
        parser = parse_llm_response_multiclass
        Y_TRUE = Y_TRUE_MULTI

    # ---- Inferencia em batches ----
    print(f"[{model_id}/{task}] Classificando {len(TEXTS):,} textos "
          f"(batch_size={BATCH_SIZE})...")
    t0 = time.perf_counter()
    raw_results = classify_batch_hf(
        tokenizer, model,
        texts=TEXTS, record_ids=RECORD_IDS,
        method=model_key,
        max_new_tokens=MAX_NEW_TOKENS,
        max_input_length=MAX_INPUT_LENGTH,
        temperature=TEMPERATURE,
        batch_size=BATCH_SIZE,
        checkpoint_path=checkpoint,
        checkpoint_every=200,
        prompt_builder=prompt_builder,
        parser=parser,
    )
    inference_time = time.perf_counter() - t0
    n_errors = sum(1 for r in raw_results if r.get("label") == "erro")
    print(f"[{model_id}/{task}] {len(raw_results)} predicoes em {inference_time:.0f}s "
          f"({n_errors} erros de parse)")

    # ---- Converter para predictions + metricas (por tarefa) ----
    if task == "binary":
        preds = hf_results_to_predictions(raw_results)
        truth = pd.DataFrame({"index": RECORD_IDS, "y_true": Y_TRUE})
        preds = preds.merge(truth, on="index", how="left")
        preds = preds[["index", "y_true", "y_pred", "y_score", "method", "label", "justificativa"]]
        valid = preds.dropna(subset=["y_true"])
        metrics = compute_binary_metrics(valid["y_true"].values, valid["y_pred"].values)
        metrics["auc_roc"] = round(
            compute_roc_auc(valid["y_true"].values, valid["y_score"].values), 4,
        )
        metrics["coverage"] = round(len(valid) / len(test_eval), 4)
        metrics_print = f"f1={metrics['f1']:.4f}  prec={metrics['precision']:.4f}  rec={metrics['recall']:.4f}  cov={metrics['coverage']:.4f}"
    else:
        preds = hf_results_to_multiclass_predictions(raw_results)
        truth = pd.DataFrame({"index": RECORD_IDS, "y_true": Y_TRUE})
        preds = preds.merge(truth, on="index", how="left")
        preds = preds[["index", "y_true", "y_pred", "method", "label", "justificativa"]]
        valid = preds.dropna(subset=["y_true"])
        metrics = compute_multiclass_metrics(
            valid["y_true"], valid["y_pred"], labels=MULTI_LABELS,
        )
        metrics["coverage"] = round(len(valid) / len(test_eval), 4)
        # Confusion matrix
        cm = compute_confusion_matrix(valid["y_true"], valid["y_pred"], labels=MULTI_LABELS)
        cm.to_csv(run_dir / "confusion_matrix.csv")
        metrics_print = f"macro_f1={metrics['macro_f1']:.4f}  acc={metrics['accuracy']:.4f}  cov={metrics['coverage']:.4f}"

    preds.to_csv(run_dir / "predictions.csv", index=False)

    cost = compute_cost_metrics(
        train_seconds=0.0,  # zero-shot
        inference_seconds=inference_time,
        n_inference_samples=len(test_eval),
        model_size_mb=model_size_mb,
        n_parameters=int(n_params),
        hardware=HARDWARE,
    )
    cost["model_load_seconds"] = round(load_time, 1)
    cost["batch_size"] = BATCH_SIZE
    cost["max_new_tokens"] = MAX_NEW_TOKENS

    config = {
        "model_name": model_name,
        "dtype": LOAD_DTYPE,
        "batch_size": BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_input_length": MAX_INPUT_LENGTH,
        "temperature": TEMPERATURE,
        "max_samples": MAX_SAMPLES,
        "task": task,
    }

    persist_result_card(build_result_card(
        model_id=model_id, task=task, regime="test_set",
        metrics=metrics, cost=cost, config=config,
        n_train_samples=0, n_eval_samples=len(test_eval),
        predictions_path=str(run_dir / "predictions.csv"),
        notes=("Zero-shot LLM classification on test set. "
               "Multiclasse: prompt lista 8 categorias editoriais. "
               "y_score binario (0/1) — LLMs nao expoem probabilidades calibradas."),
    ), run_dir)

    print(f"[{model_id}/{task}] {metrics_print}")

    # Liberar VRAM
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "model_id": model_id, "task": task,
        "metrics": metrics, "cost": cost, "n_errors": n_errors,
    }


## 6. Loop sobre todas as combinacoes

Falhas em uma combinacao (ex: GatedRepoError sem HF_TOKEN) nao matam o resto — o loop continua com o proximo `(modelo, tarefa)`.

In [ ]:
all_summaries = []
for model_key, model_name in LLMS:
    for task in TASKS:
        try:
            summary = evaluate_llm(model_key, model_name, task)
            all_summaries.append(summary)
        except Exception as exc:
            print(f"\nFALHA em {model_key}/{task}: {type(exc).__name__}: {exc}")
            print("Continuando com proximo...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

print("\n\n=== TODAS AS COMBINACOES (modelo x tarefa) AVALIADAS ===")
for s in all_summaries:
    primary = s["metrics"].get("f1") or s["metrics"].get("macro_f1")
    print(f"{s['model_id']:35s} task={s['task']:10s}  primary={primary:.4f}  "
          f"errors={s['n_errors']}  cov={s['metrics'].get('coverage'):.4f}")


## 7. Sumario comparativo

Tabela com 4 linhas (2 modelos x 2 tarefas).

In [ ]:
rows = []
for sub in sorted(RUNS_BASE.glob("llm_*_test_set")):
    card_path = sub / "result_card.json"
    if not card_path.exists():
        continue
    c = json.loads(card_path.read_text())
    if c.get("task") not in ("binary", "multiclass"):
        continue
    m = c["metrics"]
    primary = m.get("f1") or m.get("macro_f1")
    rows.append({
        "model_id": c["model_id"],
        "task": c["task"],
        "primary": primary,
        "coverage": m.get("coverage"),
        "inf_s": c["cost"].get("inference_seconds_total") or c["cost"].get("inference_seconds_mean"),
        "load_s": c["cost"].get("model_load_seconds"),
        "size_mb": c["cost"].get("model_size_mb"),
        "params_b": round((c["cost"].get("n_parameters") or 0) / 1e9, 2),
    })
summary_df = pd.DataFrame(rows).sort_values(["task", "primary"], ascending=[True, False]).reset_index(drop=True)
summary_df
